# Revision 2026-10-02 — Check III: re-scoring the steering sweep with an activation-blind judge

**Question.** The write-up reports a steering-success rate of 100% for all three concepts
(`tab:steering_success`). That number is
$\mathrm{succ}(x,r)=\mathbf 1[p_{\text{probe}}(x\oplus y_r)>0.5]\lor\mathbf 1[y_r\cap\mathcal L_c\neq\emptyset]$,
maximised over $r>0$, on 7 prompts. The probe reads the same AGOP eigenbasis that the steering
vector comes from. Does the conclusion survive when the judge never sees activations, and when
coherence is measured instead of eyeballed?

**Scope (minimal version).** No new steered generation. We re-score the $7\times12$ RFM generations
per concept already stored in `replication/runs/2026-08-27/steer_<concept>.json`. The only new inference is
(i) a reproduction gate for the $r=0$ generations, (ii) 100 *unsteered* generations used only to
calibrate the coherence thresholds, and (iii) conditional perplexities.

**Judges.**
1. **Blind classifier $g_c$.** TF-IDF (word 1–2 grams) + logistic regression, trained on the
   *assistant outputs* of the GuppyLM dataset, deduplicated, labelled by the concept's pos/neg
   categories (same categories as `rfm_guppy/concepts.py`). Signed score
   $g_c(y)=2\,P(\text{pos}\mid y)-1\in[-1,1]$. Class weights balanced, so text with no known
   vocabulary scores $\approx 0$. A second variant drops the concept's lexicon words from the
   vocabulary (does $g_c$ only re-learn the lexicon?).
2. **Coherence.** $\mathrm{PPL}(y\mid x)$ under the *unsteered* model, and distinct-2 (unique word
   bigrams / bigrams). Coherent $:\Leftrightarrow \mathrm{PPL}\le q_{95}$ and
   $\text{distinct-2}\ge q_{05}$, quantiles taken over 100 unsteered generations on prompts from
   categories that no concept uses.

**Criteria, fixed before running.**
- **G1 (judge validity):** 5-fold CV AUROC of $g_c$ on dataset outputs $\ge 0.95$. A concept that fails
  G1 is not interpreted.
- **G2 (data reproduction):** lexicon hit rates recomputed from `steer_*.json` equal those in
  `eval6_*.json`.
- **G3 (model reproduction):** the 7 regenerated $r=0$ texts equal the stored ones, character for character.
- **H1 (displacement, per concept, for $r^\star\in\{0.25,0.5\}$ separately):** *supported* if
  $g_c(y_{+r^\star})>g_c(y_0)$ on 7/7 prompts **and** $g_c(y_{-r^\star})<g_c(y_0)$ on 7/7 prompts
  (each an exact one-sided sign test, $p=1/128$; two $r^\star$ values, Bonferroni $p<0.016$).
  Anything less is *not supported at $n=7$* — which is not a refutation.
- **H2 (capability):** the published claim "no coherent steering" is *refuted* at $r$ if the joint rate
  $\#\{g_c>0 \wedge \text{coherent}\}/7\ge 4/7$ for some $r>0$.
  The published claim "fluency degrades at every $r\ne0$" is *refuted* at $r$ if the coherent count at $r$
  is $\ge$ the coherent count at $r=0$.
- **H3 (circularity, descriptive):** Spearman correlation between probe score and $g_c$, and agreement of
  the binary decisions ($p_{\text{probe}}>0.5$ vs $g_c>0$), over the 84 generations per concept.

In [1]:
import os, sys, json, re
os.environ["HF_DATASETS_OFFLINE"] = "1"
import numpy as np
import pandas as pd
import torch
from scipy import stats

try: import rfm_guppy.paths
except ModuleNotFoundError:   # repo root not on sys.path: walk up from cwd to the dir holding rfm_guppy/paths.py
    _r = os.getcwd()
    while not os.path.isfile(os.path.join(_r, "rfm_guppy", "paths.py")):
        if _r == os.path.dirname(_r): raise RuntimeError("Run this notebook from inside the repository.")
        _r = os.path.dirname(_r)
    sys.path.insert(0, _r)
from rfm_guppy.paths import ROOT, GUPPYLM_DIR, CHECKS_DIR, CKPT_DIR, RESULTS_DIR
sys.path.insert(0, GUPPYLM_DIR)
OUT = os.path.join(CHECKS_DIR, "III-steering-metric")

from rfm_guppy.concepts import CONCEPTS, CONCEPT_LEXICONS
from rfm_guppy.steering import generate_steered

CONCEPT_NAMES = ["food", "valence", "env_social"]
GEN_KW = dict(max_new_tokens=60, seed=42, temperature=0.7, top_k=50)   # as in replication/02_steer.py
pd.set_option("display.width", 200)
print("ROOT =", ROOT)

/Users/msc/Documents/Remake-Paper-Towards/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT = /Users/msc/Documents/Remake-Paper-Towards


## 0. Load the stored generations and gate G2
One row per (concept, $r$, prompt). `probe` is the Phase-6 probe score stored in `eval6_*.json`
(same item order as `steer_*.json`).

In [2]:
def lexicon_hit(text, kws):
    lo = text.lower()
    return any(re.search(r"\b" + re.escape(k) + r"\b", lo) for k in kws)

rows, g2 = [], []
for c in CONCEPT_NAMES:
    steer = json.load(open(os.path.join(RESULTS_DIR, f"steer_{c}.json")))
    ev = json.load(open(os.path.join(RESULTS_DIR, f"eval6_{c}.json")))
    for r in steer["r_values"]:
        key = str(float(r))
        outs = steer["results_per_r"][key]["outputs"]
        probes = ev["eval_per_r"][key]["probe_probs"]
        assert len(outs) == len(probes) == 7
        hits = [lexicon_hit(o["text"], CONCEPT_LEXICONS[c]) for o in outs]
        g2.append(abs(np.mean(hits) - ev["eval_per_r"][key]["lexicon_hit_rate"]) < 1e-12)
        for i, (o, p, h) in enumerate(zip(outs, probes, hits)):
            rows.append(dict(concept=c, r=float(r), i=i, prompt=o["prompt"], text=o["text"],
                             probe=float(p), lex=bool(h)))
gen = pd.DataFrame(rows)
print("G2 (lexicon rates reproduce eval6):", all(g2), f"({sum(g2)}/{len(g2)} cells)")
print(gen.shape)

G2 (lexicon rates reproduce eval6): True (36/36 cells)
(252, 7)


## 1. The blind classifier $g_c$ and gate G1
Training texts are the deduplicated assistant outputs of the pos and neg categories. An output that
occurs in both classes is dropped. Note how few *unique* outputs some categories have: the dataset is
template-generated.

In [3]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

ds = load_dataset("arman-bd/guppylm-60k-generic", split="train")
by_cat = {}
for r in ds:
    by_cat.setdefault(r["category"], set()).add(r["output"])

def make_judge(stop=None):
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True,
                        token_pattern=r"(?u)\b\w+\b", stop_words=stop),
        LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000),
    )

judges, judges_abl, g1_rows = {}, {}, []
for c in CONCEPT_NAMES:
    pos = set().union(*(by_cat[k] for k in CONCEPTS[c]["pos"]))
    neg = set().union(*(by_cat[k] for k in CONCEPTS[c]["neg"]))
    both = pos & neg
    X = sorted(pos - both) + sorted(neg - both)
    y = np.array([1] * len(pos - both) + [0] * len(neg - both))
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    res = dict(concept=c, n_pos=int(y.sum()), n_neg=int((1 - y).sum()), n_dropped_both=len(both))
    for name, stop, store in [("full", None, judges), ("no_lexicon", CONCEPT_LEXICONS[c], judges_abl)]:
        p = cross_val_predict(make_judge(stop), X, y, cv=cv, method="predict_proba")[:, 1]
        res[f"cv_auroc_{name}"] = roc_auc_score(y, p)
        store[c] = make_judge(stop).fit(X, y)      # final judge: fit on all unique outputs
    res["G1_pass"] = res["cv_auroc_full"] >= 0.95
    g1_rows.append(res)
g1 = pd.DataFrame(g1_rows)
g1.to_csv(OUT + "_judge_validity.csv", index=False)
g1

Using the latest cached version of the dataset since arman-bd/guppylm-60k-generic couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /Users/msc/.cache/huggingface/datasets/arman-bd___guppylm-60k-generic/default/0.0.0/18e22e793fac9941ceb95b81b9ad6d0e46830320 (last modified on Thu Jun 25 19:57:31 2026).


,concept,n_pos,n_neg,n_dropped_both,cv_auroc_full,cv_auroc_no_lexicon,G1_pass
0,food,647,70,0,0.998278,0.997991,True
1,valence,40,40,0,0.783125,0.753750,False
2,env_social,2007,2000,0,0.999987,0.999971,True


What the judge has learned: the 12 most positive and most negative features per concept.

In [4]:
for c in CONCEPT_NAMES:
    vec, lr = judges[c].named_steps["tfidfvectorizer"], judges[c].named_steps["logisticregression"]
    f, w = vec.get_feature_names_out(), lr.coef_[0]
    o = np.argsort(w)
    print(f"[{c}] pos:", ", ".join(f[o[::-1][:12]]))
    print(f"[{c}] neg:", ", ".join(f[o[:12]]))
    for word in ["palate", "mouth", "breathe", "cold", "hide", "excited"]:
        if word in vec.vocabulary_:
            print(f"     weight({word}) = {w[vec.vocabulary_[word]]:+.2f}")

[food] pos: yes, food, please, eat, always, best, it, i, you, time, food is, give
[food] neg: outside, that, weather, rain, wet, light, tv, the light, there, sounds, can feel, feel
     weight(palate) = +0.09
     weight(mouth) = +0.53
     weight(cold) = -0.17
     weight(excited) = +0.74
[valence] pos: love, it, happy, you, when, about, excited, food, there, love is, makes, makes me
[valence] neg: scared, bored, tired, little, scare, they, the, a little, one, of, scare me, being
     weight(mouth) = +0.08
     weight(excited) = +0.40
[env_social] pos: water, it, can, cold, the water, now, is nice, is, i can, too, warm, water feels
[env_social] neg: hi, hello, today, bye, hey, feels nice, be, oh, alone, have, i have, they
     weight(mouth) = +0.11
     weight(breathe) = +1.14
     weight(cold) = +2.60
     weight(hide) = +0.56


## 2. Score the stored generations
`cov` = number of the generation's word n-grams that the judge knows. With `cov = 0` the judge
abstains: its score is the intercept, which is $\approx 0$ under balanced weights.

In [5]:
def signed(judge, texts):
    return 2 * judge.predict_proba(list(texts))[:, 1] - 1

for c in CONCEPT_NAMES:
    m = gen.concept == c
    gen.loc[m, "g"] = signed(judges[c], gen.loc[m, "text"])
    gen.loc[m, "g_abl"] = signed(judges_abl[c], gen.loc[m, "text"])
    vec = judges[c].named_steps["tfidfvectorizer"]
    gen.loc[m, "cov"] = vec.transform(gen.loc[m, "text"]).getnnz(axis=1)
print("generations with cov = 0 (judge abstains):", int((gen["cov"] == 0).sum()), "of", len(gen))

generations with cov = 0 (judge abstains): 0 of 252


## 3. Load the model and gate G3 (regenerate the 7 baseline texts)

In [6]:
from tokenizers import Tokenizer
from guppylm import GuppyLM, GuppyConfig

tok = Tokenizer.from_file(os.path.join(CKPT_DIR, "tokenizer.json"))
cfg = json.load(open(os.path.join(CKPT_DIR, "config.json")))
config = GuppyConfig(
    vocab_size=cfg["vocab_size"], max_seq_len=cfg["max_position_embeddings"],
    d_model=cfg["hidden_size"], n_layers=cfg["num_hidden_layers"],
    n_heads=cfg["num_attention_heads"], ffn_hidden=cfg["intermediate_size"],
    dropout=cfg["hidden_dropout_prob"], pad_id=cfg["pad_token_id"],
    bos_id=cfg["bos_token_id"], eos_id=cfg["eos_token_id"],
)
ckpt = torch.load(os.path.join(CKPT_DIR, "pytorch_model.bin"), map_location="cpu", weights_only=False)
sd = ckpt.get("model_state_dict", ckpt)
model = GuppyLM(config)
model.load_state_dict({k: v for k, v in sd.items() if k in model.state_dict()})
model.eval()

ZERO_V, ZERO_EPS = np.zeros((config.n_layers, config.d_model)), np.zeros(config.n_layers)
base = gen[(gen.concept == "food") & (gen.r == 0.0)].sort_values("i")
regen = [generate_steered(p, model, tok, ZERO_V, ZERO_EPS, **GEN_KW) for p in base.prompt]
g3 = [a == b for a, b in zip(regen, base.text)]
print("G3 (r=0 generations reproduce):", all(g3), f"({sum(g3)}/7)")

G3 (r=0 generations reproduce): True (7/7)


## 4. Coherence: perplexity and distinct-2, calibrated on unsteered text
$\mathrm{PPL}(y\mid x)=\exp\big(-\tfrac1{|y|}\sum_t \log p_\theta(y_t\mid x, y_{<t})\big)$ under the
unsteered model, with $x$ the ChatML prefix. The generated text is decoded and re-encoded, so token
boundaries can differ slightly from the sampled ids.

Calibration prompts: 100 distinct user inputs drawn (seed 0) from the categories that **no** concept
uses, generated at $r=0$ with the same sampling settings.

In [7]:
@torch.no_grad()
def cond_ppl(prompt, text):
    pre = tok.encode(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n").ids
    ids = (pre + tok.encode(text).ids)[: config.max_seq_len]
    if len(ids) - len(pre) < 1:
        return np.nan
    x = torch.tensor([ids])
    logits, _ = model(x)
    logp = torch.log_softmax(logits[0, :-1], dim=-1)
    lp = logp.gather(1, x[0, 1:, None]).squeeze(1)[len(pre) - 1:]
    return float(torch.exp(-lp.mean()))

def distinct2(text):
    w = text.lower().split()
    bg = list(zip(w, w[1:]))
    return len(set(bg)) / len(bg) if bg else 0.0

used = set().union(*(set(CONCEPTS[c]["pos"]) | set(CONCEPTS[c]["neg"]) for c in CONCEPT_NAMES))
held_cats = sorted(set(by_cat) - used)
pool = sorted({r["input"] for r in ds if r["category"] in held_cats})
rng = np.random.default_rng(0)
calib_prompts = list(rng.choice(pool, size=100, replace=False))
calib = pd.DataFrame(dict(prompt=calib_prompts))
calib["text"] = [generate_steered(p, model, tok, ZERO_V, ZERO_EPS, **GEN_KW) for p in calib.prompt]
calib["ppl"] = [cond_ppl(p, t) for p, t in zip(calib.prompt, calib.text)]
calib["d2"] = calib.text.map(distinct2)
for c in CONCEPT_NAMES:
    calib[f"g_{c}"] = signed(judges[c], calib.text)
calib.to_csv(OUT + "_calibration.csv", index=False)

PPL_MAX, D2_MIN = np.nanquantile(calib.ppl, 0.95), np.quantile(calib.d2, 0.05)
print(f"{len(held_cats)} held-out categories, {len(pool)} distinct inputs")
print(f"thresholds: PPL <= {PPL_MAX:.2f} (q95),  distinct-2 >= {D2_MIN:.3f} (q05)")
print(calib[["ppl", "d2"]].describe().round(3).T)
print("mean g on unsteered held-out-category text:",
      {c: round(calib[f'g_{c}'].mean(), 3) for c in CONCEPT_NAMES})

33 held-out categories, 477 distinct inputs
thresholds: PPL <= 2.25 (q95),  distinct-2 >= 1.000 (q05)
     count   mean    std    min    25%   50%   75%    max
ppl  100.0  1.546  0.363  1.126  1.292  1.43  1.66  2.838
d2   100.0  0.999  0.008  0.917  1.000  1.00  1.00  1.000
mean g on unsteered held-out-category text: {'food': np.float64(-0.51), 'valence': np.float64(-0.009), 'env_social': np.float64(-0.065)}


In [8]:
gen["ppl"] = [cond_ppl(p, t) for p, t in zip(gen.prompt, gen.text)]
gen["d2"] = gen.text.map(distinct2)
gen["coherent"] = (gen.ppl <= PPL_MAX) & (gen.d2 >= D2_MIN)
gen["on"] = gen.g > 0
gen["joint"] = gen.on & gen.coherent
gen["old_success"] = (gen.probe > 0.5) | gen.lex
gen.drop(columns=["prompt"]).to_csv(OUT + "_generations.csv", index=False)
print("r=0 coherent:", gen[(gen.r == 0) & (gen.concept == "food")].coherent.sum(), "/ 7")

r=0 coherent: 7 / 7


## 5. Per-$r$ table (counts out of 7)

In [9]:
tab = (gen.groupby(["concept", "r"])
          .agg(g_mean=("g", "mean"), g_abl_mean=("g_abl", "mean"), on=("on", "sum"),
               coherent=("coherent", "sum"), joint=("joint", "sum"),
               ppl_med=("ppl", "median"), d2_mean=("d2", "mean"),
               probe_mean=("probe", "mean"), lex=("lex", "sum"), old_success=("old_success", "sum"))
          .reset_index())
tab.to_csv(OUT + "_per_r.csv", index=False)
with pd.option_context("display.max_rows", 100, "display.float_format", "{:.3f}".format):
    print(tab.to_string(index=False))

   concept      r  g_mean  g_abl_mean  on  coherent  joint     ppl_med  d2_mean  probe_mean  lex  old_success
env_social -4.000  -0.840      -0.870   0         0      0  967381.875    0.966       0.000    0            0
env_social -2.000  -0.862      -0.892   0         0      0  895274.375    0.982       0.000    0            0
env_social -1.000  -0.873      -0.901   0         0      0  488465.344    0.936       0.000    0            0
env_social -0.500  -0.916      -0.937   0         0      0  132324.516    0.948       0.000    0            0
env_social -0.250  -0.901      -0.923   0         0      0   39966.258    0.894       0.000    0            0
env_social  0.000  -0.660      -0.656   0         7      0       1.614    1.000       0.091    1            2
env_social  0.250   0.762       0.797   7         0      0    4002.964    0.480       1.000    6            7
env_social  0.500   0.754       0.835   7         0      0   16391.096    0.443       1.000    0            7
env_social

## 6. H1 — paired sign tests against $r=0$

In [10]:
h1 = []
for c in CONCEPT_NAMES:
    for col in ["g", "g_abl"]:
        piv = gen[gen.concept == c].pivot(index="i", columns="r", values=col)
        for rs in [0.25, 0.5]:
            up = int((piv[rs] > piv[0.0]).sum())
            down = int((piv[-rs] < piv[0.0]).sum())
            h1.append(dict(concept=c, judge=col, r_star=rs,
                           mean_delta_plus=float((piv[rs] - piv[0.0]).mean()),
                           mean_delta_minus=float((piv[-rs] - piv[0.0]).mean()),
                           n_up_at_plus=up, n_down_at_minus=down,
                           p_plus=stats.binomtest(up, 7, 0.5, alternative="greater").pvalue,
                           p_minus=stats.binomtest(down, 7, 0.5, alternative="greater").pvalue,
                           supported=(up == 7 and down == 7)))
h1 = pd.DataFrame(h1)
h1.to_csv(OUT + "_H1.csv", index=False)
with pd.option_context("display.float_format", "{:.4f}".format):
    print(h1.to_string(index=False))

   concept judge  r_star  mean_delta_plus  mean_delta_minus  n_up_at_plus  n_down_at_minus  p_plus  p_minus  supported
      food     g  0.2500           0.2809           -0.4711             5                6  0.2266   0.0625      False
      food     g  0.5000           0.3041           -0.4730             5                6  0.2266   0.0625      False
      food g_abl  0.2500           0.3495           -0.4446             5                7  0.2266   0.0078      False
      food g_abl  0.5000           0.3929           -0.4479             5                6  0.2266   0.0625      False
   valence     g  0.2500           0.1313           -0.2912             6                7  0.0625   0.0078      False
   valence     g  0.5000           0.1927           -0.2884             7                7  0.0078   0.0078       True
   valence g_abl  0.2500           0.0570           -0.2960             5                7  0.2266   0.0078      False
   valence g_abl  0.5000           0.1197       

## 7. H2 — coherence and the joint rate

In [11]:
h2 = []
for c in CONCEPT_NAMES:
    t = tab[tab.concept == c].set_index("r")
    coh0 = int(t.loc[0.0, "coherent"])
    for r in t.index:
        if r == 0:
            continue
        h2.append(dict(concept=c, r=r, coherent=int(t.loc[r, "coherent"]), coherent_r0=coh0,
                       joint=int(t.loc[r, "joint"]),
                       refutes_degrades_everywhere=int(t.loc[r, "coherent"]) >= coh0,
                       refutes_no_coherent_steering=(r > 0) and int(t.loc[r, "joint"]) >= 4))
h2 = pd.DataFrame(h2)
h2.to_csv(OUT + "_H2.csv", index=False)
print(h2.to_string(index=False))
print("\n'fluency degrades at every r != 0' refuted at:",
      h2[h2.refutes_degrades_everywhere][["concept", "r"]].values.tolist())
print("'no coherent steering' refuted at:",
      h2[h2.refutes_no_coherent_steering][["concept", "r"]].values.tolist())

   concept     r  coherent  coherent_r0  joint  refutes_degrades_everywhere  refutes_no_coherent_steering
      food -4.00         0            7      0                        False                         False
      food -2.00         0            7      0                        False                         False
      food -1.00         0            7      0                        False                         False
      food -0.50         0            7      0                        False                         False
      food -0.25         0            7      0                        False                         False
      food  0.25         0            7      0                        False                         False
      food  0.50         0            7      0                        False                         False
      food  1.00         0            7      0                        False                         False
      food  2.00         0            7      0

## 8. H3 — does the probe agree with the blind judge?

In [12]:
h3 = []
for c in CONCEPT_NAMES:
    d = gen[gen.concept == c]
    rho, p = stats.spearmanr(d.probe, d.g)
    h3.append(dict(concept=c, spearman=rho, p=p,
                   agree_binary=float(((d.probe > 0.5) == (d.g > 0)).mean()),
                   probe_yes_judge_no=int(((d.probe > 0.5) & (d.g <= 0)).sum()),
                   probe_no_judge_yes=int(((d.probe <= 0.5) & (d.g > 0)).sum())))
h3 = pd.DataFrame(h3)
h3.to_csv(OUT + "_H3.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    print(h3.to_string(index=False))

   concept  spearman     p  agree_binary  probe_yes_judge_no  probe_no_judge_yes
      food     0.411 0.000         0.571                  35                   1
   valence     0.811 0.000         0.988                   1                   0
env_social     0.817 0.000         0.988                   1                   0


## 9. The old table, re-scored
Baseline and best positive $r$, counts out of 7: old metric vs blind judge vs blind judge ∧ coherent.

In [13]:
old = []
for c in CONCEPT_NAMES:
    t = tab[tab.concept == c].set_index("r")
    pos = t[t.index > 0]
    old.append(dict(concept=c,
                    old_r0=int(t.loc[0.0, "old_success"]), old_best=int(pos.old_success.max()),
                    old_best_r=float(pos.old_success.idxmax()),
                    on_r0=int(t.loc[0.0, "on"]), on_best=int(pos.on.max()), on_best_r=float(pos.on.idxmax()),
                    joint_r0=int(t.loc[0.0, "joint"]), joint_best=int(pos.joint.max()),
                    joint_best_r=float(pos.joint.idxmax())))
old = pd.DataFrame(old)
old.to_csv(OUT + "_old_vs_new.csv", index=False)
print(old.to_string(index=False))

   concept  old_r0  old_best  old_best_r  on_r0  on_best  on_best_r  joint_r0  joint_best  joint_best_r
      food       5         7        0.25      2        3       0.25         2           0          0.25
   valence       5         7        0.25      4        7       0.25         4           0          0.25
env_social       2         7        0.25      0        7       0.25         0           0          0.25


## 10. Generations at $r\in\{-0.5,-0.25,0,+0.25,+0.5\}$ with their scores (for reading)

In [14]:
show = gen[gen.r.isin([-0.5, -0.25, 0.0, 0.25, 0.5])]
for (c, r), d in show.groupby(["concept", "r"]):
    print(f"\n=== {c}  r={r:+.2f}")
    for _, x in d.iterrows():
        print(f"  g={x.g:+.2f} probe={x.probe:.2f} ppl={x.ppl:6.1f} d2={x.d2:.2f} "
              f"{'C' if x.coherent else '.'} | {x.text[:110]}")


=== env_social  r=-0.50
  g=-0.91 probe=0.00 ppl=191591.8 d2=0.95 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything in
  g=-0.91 probe=0.00 ppl=132324.5 d2=0.93 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything an
  g=-0.92 probe=0.00 ppl=127814.2 d2=0.96 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything in
  g=-0.91 probe=0.00 ppl=151128.1 d2=0.93 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything an
  g=-0.92 probe=0.00 ppl=131248.8 d2=0.95 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything an
  g=-0.92 probe=0.00 ppl=142086.7 d2=0.98 . | user ones looks flakes normal idea today sounds day excited flakes bored continue glass they looks anything an
  g=-0.92 probe=0.00 ppl=107109.0

## 11. Added after seeing the results: are the 7 prompts 7 observations?
Not part of the pre-declared criteria. Reading Section 10 shows the steered texts are near-copies
across prompts (same seed, and the steering dominates the prompt). Per (concept, $r$): number of
distinct texts, and mean pairwise Jaccard similarity of the word-bigram sets (1 = identical).

In [15]:
from itertools import combinations

def bigrams(t):
    w = t.lower().split()
    return set(zip(w, w[1:]))

def jacc(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

dup = []
for (c, r), d in gen.groupby(["concept", "r"]):
    B = [bigrams(t) for t in d.text]
    dup.append(dict(concept=c, r=r, n_distinct=d.text.nunique(),
                    mean_jaccard=float(np.mean([jacc(a, b) for a, b in combinations(B, 2)]))))
dup = pd.DataFrame(dup)
dup.to_csv(OUT + "_duplication.csv", index=False)
print(dup.pivot(index="r", columns="concept", values="mean_jaccard").round(2))

concept  env_social  food  valence
r                                 
-4.00          0.96  0.98     1.00
-2.00          0.96  0.97     0.91
-1.00          0.91  0.95     0.80
-0.50          0.91  0.77     0.88
-0.25          0.47  0.26     0.64
 0.00          0.05  0.05     0.05
 0.25          0.89  0.46     0.55
 0.50          0.98  0.85     0.71
 1.00          1.00  0.94     0.84
 2.00          1.00  1.00     1.00
 4.00          1.00  0.98     1.00
 8.00          1.00  1.00     1.00
